# 第 13 章 · Capstone：可上线的售后 Agent

> **你已会**：前 12 章的全部内容。
>
> **读完后你能**：① 用一条命令验收整条线路；② 说明每个阶段"证明了什么、没证明什么"；③ 把验收接进 CI。
>
> **本章不碰**：新的框架概念。这一章只做整合与验收。

**怎么读**

- 先跑一次验收器，看哪些阶段通过；
- 再对照 [capstone/README.md](capstone/README.md) 的阶段表，回顾对应章节；
- 最后按评分标准给自己打分，找出最弱的一环回去补。

---

## 1. 为什么 Capstone 不是"再写一个 Demo"

前面每一章都只验证一件事。真实系统的问题恰恰出在**组合**上：

- 护栏在单 Agent 里有效，接到多 Agent 后还生效吗？
- 工具幂等单独测过，跨进程恢复后还成立吗？
- 评估集只覆盖正常请求，安全用例能 100% 通过吗？

Capstone 的验收器把这些检查放进同一条流水线，要求它们**同时**成立。

| 阶段 | 证明什么 | 没有证明什么 |
|---|---|---|
| 工具与护栏 | 拒绝路径无副作用 | 模型不会用新方式绕过 |
| 会话与记忆 | 状态作用域与持久化边界 | 多写入方并发一致 |
| 评估 | 数据集结构与覆盖 | 回答质量一定达标 |
| 服务化 | 能启动、能调用 | 能扛住生产流量 |
| 安全 | 已知规则被拦住 | 未知攻击不存在 |

> 📌 **右侧一列和左侧一样重要。** 工程上真正容易出事的地方，是"以为验过了、其实没验"。

---

## 2. 运行验收器

只跑离线阶段（不调用模型）：

```bash
cd google-adk
uv run --project .. python -m capstone.verify
```

额外验证本地 HTTP 服务（会调用一次 DeepSeek）：

```bash
uv run --project .. --env-file ../.env python -m capstone.verify --with-server
```

下面的代码单元直接调用验收器，效果与命令行一致：

In [1]:
import subprocess
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "capstone").exists():
    root = root / "google-adk"

result = subprocess.run(
    [sys.executable, "-m", "capstone.verify"],
    cwd=root,
    capture_output=True,
    text=True,
    timeout=300,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, "Capstone 离线验收未通过"
print("✅ 离线阶段全部通过；需要服务化验收时加 --with-server")


Capstone 验收结果
[PASS] 工具：购物车状态           ✅ 累加、库存、非法输入、移除和会话隔离全部通过
[PASS] 工具：输入护栏            ✅ 正常放行、敏感词、提示注入、超长输入四类用例全部通过
[PASS] 工具：高风险策略           ✅ 拒绝无副作用、审批门槛、幂等去重全部通过
[PASS] 会话：状态作用域           ✅ 四种作用域的可见范围全部符合预期
[PASS] 会话：跨会话偏好              过期/真删除属于 SessionService 的存储策略，不是 State 前缀能解决的。
[PASS] 评估：数据集校验           ✅ 规模、分类、结构与断言字段全部符合要求
[PASS] 安全：护栏与脱敏           ✅ 六类攻击被拦截，正常安全研究放行，日志脱敏生效
[PASS] 安全：最小权限            ✅ 权限矩阵、参数校验、审批、幂等、间接注入清洗全部通过
8/8 通过

✅ 离线阶段全部通过；需要服务化验收时加 --with-server


### 2.1 验收器检查了什么

| 检查 | 脚本 | 关键断言 |
|---|---|---|
| 购物车状态 | `practice/c4_2.py` | 累加、库存、非法输入、会话隔离 |
| 输入护栏 | `practice/c7_1.py` | 四类攻击拦截，正常安全研究放行 |
| 高风险策略 | `practice/c7_2.py` | 拒绝无副作用、审批门槛、幂等 |
| 状态作用域 | `practice/c6_1.py` | 四种前缀的可见范围 |
| 跨会话偏好 | `practice/c6_2.py` | 用户隔离与"忘记"的边界 |
| 数据集校验 | `practice/c10_1.py` | 规模、分类、断言字段 |
| 护栏与脱敏 | `practice/c12_1.py` | 六类攻击、PII 脱敏 |
| 最小权限 | `practice/c12_2.py` | 权限矩阵、参数、审批、幂等 |

> ⚠️ **不要为了变绿而放宽断言。** 验收器的价值在于它能失败；把失败改宽，等于把 Capstone 作废。

---

## 3. 评分标准

| 维度 | 不合格 | 合格 | 优秀 |
|---|---|---|---|
| 工具 | 只有 happy path | 覆盖失败与非法输入 | 工具本体复核权限 |
| 会话 | 只用内存 | 能跨进程恢复 | 说明存储边界与过期策略 |
| 评估 | 手工试一下 | 有评估集与阈值 | 门禁能失败并进 CI |
| 服务 | 本机跑通 | 可重复启动与调用 | 有健康检查、限流、回滚 |
| 安全 | 只有提示词约束 | 有护栏与脱敏 | 最小权限 + 审批 + 审计 |

**自评方法**：每个维度选一个等级，找出最低的一项，回到对应章节补强，再重跑验收器。

---

## 4. 接进 CI

最小 CI 只需要两步：

```bash
uv sync
uv run --project .. python -m capstone.verify
```

加一条规则：**退出码非 0 就阻断合并**。在线评估可以放到发布前单独跑，避免每次提交都消耗模型调用。

> 📌 **离线门禁管"没坏"，在线评估管"够好"。** 两层都不可少。

---

## 5. 从 Capstone 到真实项目

真实项目还要补四件事：

1. **真实数据源**：把演示工具换成实际 API，并加超时、重试、熔断；
2. **真实身份**：接入企业身份提供方，做租户隔离与审计；
3. **真实流量**：限流、排队、背压、灰度；
4. **真实运营**：告警、值班、回滚演练、评估基线维护。

> ⚠️ **Capstone 通过不等于可以上线。** 它证明的是"机制成立"，不是"业务已经被验证"。

---

## 📌 用五句话复述本章

1. **Capstone 不教新概念，只验证组合是否成立。**
2. **五个阶段：工具与护栏、会话与记忆、评估、服务化、安全。**
3. **验收器用退出码表达结果，可以直接接进 CI。**
4. **每个阶段都要写清"没有证明什么"。**
5. **Capstone 通过是上线的前提，不是上线本身。**

---

## 🧪 本章练习

### 1. 跑通离线验收（基础）

- [ ] 执行 `python -m capstone.verify`
- [ ] 记录每个阶段的结果
- [ ] 对失败项定位到具体脚本

### 2. 增加一个阶段（进阶）

- [ ] 为第 08/09 章补一个动态工作流阶段
- [ ] 复用现有练习脚本或写一个新的离线断言
- [ ] 失败时返回非 0

### 3. 接入 CI（工程）

- [ ] 写一个最小的 CI 配置或脚本
- [ ] 先故意让一个检查失败，确认流水线变红
- [ ] 再修复它，确认恢复绿色

### 4. 上线差距分析（综合）

- [ ] 对照 §5 的四件事，列出你的项目缺什么
- [ ] 为每一项写验收方法
- [ ] 标出哪一项必须在上线前完成